In [ ]:
import os
import sys
import inspect
from pydantic import Field
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import START, StateGraph
from langchain.chat_models import init_chat_model
from typing import Dict, List, Literal,  Optional
from langchain_core.prompts import ChatPromptTemplate
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from utils.utility import load_config, Settings, load_prompt
import utils.tools as ToolsLlm


class State(TypedDict):
    question: str
    plan: Dict
    outputs: Dict
    response: str 

class ArgPair(TypedDict):
    key: str
    value: str

class TaskResponseFormatter(TypedDict):
    task: Literal["summary_statistics", "predict_population_linear", "calculator", "plot_population"] = Field(description="Name of the tool to run")
    id: str = Field(description="Unique identifier for the task")
    dep: List[str] = Field(description="List of task ids this task depends on")
    args: Optional[List[ArgPair]] = Field(description="Arguments as list of (key, value) pairs")

class Plan(TypedDict):
    """Always use this tool to structure your response to the user in the planning phase."""
    plan: List[TaskResponseFormatter] = Field(description="List of tasks to be execute")

class Agent(): 
    def __init__(self, settings: Settings):
        self.llm_config = settings.llm
        self.state: State = {"question":"", "plan": [], "outputs": [], "response": ""}
        self.config_prompt()
        self.config_llm()
        graph_builder = StateGraph(State).add_sequence(
            [self.ask, self.task_planning, self.validate_plan, self.run_plan()]
        )
        self.graph = graph_builder.compile()

    def config_prompt(self):
        self.planning_prompt = load_prompt("planning_stage")
        self.response_prompt = load_prompt("response_stage")
        self.planning_prompt_temp = ChatPromptTemplate([("system", self.planning_prompt), ("user", "Question: {input}")])
    
    def config_llm(self):
        self.llm = init_chat_model(model=self.llm_config.model_name, model_provider=self.llm_config.provider)
        self.plan_structure_llm = self.llm.with_structured_output(Plan)

    def ask(self, question: str):
        self.state["question"] = question
    
    def task_planning(self): 
      prompt = self.planning_prompt_temp.invoke({"input": self.state["question"],})
      result = self.plan_structure_llm.invoke(prompt)
      self.state["plan"] = result["plan"]

    def validate_plan(self):
        task_ids = set()
        errors = []
        for task in self.state["plan"]:
            task_id = task.get('id')
            task_name = task.get('task')
            deps = task.get('dep', [])
            args = task.get('args', [])

            # Check ID
            if not isinstance(task_id, str):
                errors.append(f"Task ID must be a string: {task_id}")
            elif task_id in task_ids:
                errors.append(f"Duplicate task ID found: {task_id}")
            else:
                task_ids.add(task_id)

            # Check task name
            if task_name not in ToolsLlm.TASK_FUNCS:
                errors.append(f"Invalid task name: {task_name}")

            # Check deps
            if not isinstance(deps, list):
                errors.append(f"Dependencies must be a list for task {task_id}")
            else:
                for dep_id in deps:
                    if not isinstance(dep_id, str):
                        errors.append(f"Dependency ID must be string in task {task_id}: {dep_id}")

            # Check args format
            if not isinstance(args, list):
                errors.append(f"Args must be a list for task {task_id}")
            else:
                for arg in args:
                    if not isinstance(arg, dict) or 'key' not in arg or 'value' not in arg:
                        errors.append(f"Invalid arg format in task {task_id}: {arg}")

        # validate if dependencies exist
        for task in self.state["plan"]:
            for dep_id in task.get('dep', []):
                if dep_id not in task_ids:
                    errors.append(f"Task {task['id']} has unknown dependency: {dep_id}")

            for arg in task.get('args', []):
                val = arg['value']
                if isinstance(val, str) and val.startswith("DEP_"):
                    dep_ref = val[4:]
                    if dep_ref not in task_ids:
                        errors.append(f"Task {task['id']} references unknown DEP value: {val}")

        # Validate args against function signature
        for task in self.state["plan"]:
            func = ToolsLlm.TASK_FUNCS.get(task['task'])
            if func:
                sig = inspect.signature(func)
                expected_args = set(sig.parameters.keys())
                actual_args = set(arg['key'] for arg in task['args'])
                if not actual_args.issubset(expected_args):
                    extra = actual_args - expected_args
                    errors.append(f"Task {task['id']} has unexpected args: {extra}")
                missing = expected_args - actual_args
                if missing:
                    errors.append(f"Task {task['id']} is missing args: {missing}")
        if errors:
            raise ValueError("Plan validation failed:\n" + "\n".join(errors))

    def run_plan(self): 
      outputs = {}
      def resolve_args(args_list):
          resolved = {}
          for arg in args_list:
              k = arg['key']
              v = arg['value']
              if isinstance(v, str) and v.startswith("DEP_"):
                  dep_task_id = v[4:]
                  if dep_task_id not in outputs:
                      raise ValueError(f"Dependency output for {dep_task_id} not ready")
                  resolved[k] = outputs[dep_task_id]
              else:
                  resolved[k] = v
          return resolved
      remaining = self.state["plan"].copy()
      while remaining:
          progress = False
          for task in remaining[:]: 
              if all(dep in outputs for dep in task['dep']):
                  args_list = task.get('args', []) 
                  args = resolve_args(args_list)

                  func = ToolsLlm.TASK_FUNCS.get(task['task'])
                  if not func:
                      raise ValueError(f"No function defined for task {task['task']}")
                  result = func(**args)

                  outputs[task['id']] = result
                  remaining.remove(task)
                  progress = True
          if not progress:
              raise RuntimeError("Circular dependency or missing dependencies detected")
      self.state["outputs"] = outputs
      self.generate_response()

    def generate_response(self): 
        """Function to generate the response to the user in text manner"""

settings = Settings(**load_config("config/settings.yaml"))
agent = Agent(settings)
agent.ask("Forecast the population for 2028.")

